In [1]:
import json
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Arc

REPO_ROOT = Path("../..").resolve()

NOMBRE = "A1606b0e6_0_62"
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
MARGEN_CAMPO = 5.0

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(v) for k, v in data["color_equipo_bgr"].items()}

def en_campo(c):
    return (c is not None and -MARGEN_CAMPO <= c[0] <= 105 + MARGEN_CAMPO
            and -MARGEN_CAMPO <= c[1] <= 68 + MARGEN_CAMPO)

print(f"JSON: {JSON_IN.name} | Frames: {N_FRAMES} | Vídeo existe: {VIDEO_IN.exists()}")

JSON: integrado_A1606b0e6_0_62.json | Frames: 749 | Vídeo existe: True


In [2]:
def dibujar_campo(ax):
    ax.add_patch(Rectangle((0, 0), 105, 68, color="#3a7d34", zorder=0))
    lc, lw = "white", 2
    ax.add_patch(Rectangle((0, 0), 105, 68, fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([52.5, 52.5], [0, 68], color=lc, linewidth=lw)
    ax.add_patch(Circle((52.5, 34), 9.15, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((52.5, 34), 0.4, color=lc))
    ax.add_patch(Rectangle((0, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((11, 34), 0.4, color=lc))
    ax.add_patch(Arc((11, 34), 18.3, 18.3, theta1=-53, theta2=53, color=lc, linewidth=lw))
    ax.add_patch(Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((99.5, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((94, 34), 0.4, color=lc))
    ax.add_patch(Arc((94, 34), 18.3, 18.3, theta1=127, theta2=233, color=lc, linewidth=lw))
    ax.set_xlim(-5, 110); ax.set_ylim(-5, 73)
    ax.set_aspect("equal"); ax.axis("off")

print("Función dibujar_campo definida")

Función dibujar_campo definida


In [3]:
# ---- Trayectorias por identidad: filtrar saltos, suavizar, rellenar huecos + balón interpolado ----
VENTANA = 5         # media móvil por identidad (suaviza el temblor)
MAX_HUECO = 15      # huecos de jugador más cortos se interpolan
MAX_VEL = 1.2       # m/frame: por encima = salto de homografía, no movimiento real

# Agrupar posiciones SANAS de jugadores/porteros por (id, equipo, rol)
tray = defaultdict(dict); meta = {}
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") not in ("jugador", "portero"):
            continue
        if not en_campo(d.get("campo")) or d.get("equipo") not in (0, 1):
            continue
        key = (d["id"], d["equipo"], d["rol"])
        tray[key][f_idx] = tuple(d["campo"])
        meta[key] = {"equipo": d["equipo"], "rol": d["rol"]}

def filtrar_saltos(serie):
    """Elimina posiciones con velocidad físicamente imposible (saltos de homografía)."""
    if len(serie) < 2:
        return dict(serie)
    fs = sorted(serie); limpia = {fs[0]: serie[fs[0]]}; f_ant, p_ant = fs[0], serie[fs[0]]
    for f in fs[1:]:
        p = serie[f]; vel = np.hypot(p[0]-p_ant[0], p[1]-p_ant[1]) / max(f-f_ant, 1)
        if vel <= MAX_VEL:
            limpia[f] = p; f_ant, p_ant = f, p
    return limpia

def suavizar_y_rellenar(serie):
    serie = filtrar_saltos(serie)
    if not serie:
        return {}
    fs = sorted(serie); completa = {}
    for a, b in zip(fs, fs[1:]):
        completa[a] = serie[a]; hueco = b - a
        if 1 < hueco <= MAX_HUECO:
            xa, ya = serie[a]; xb, yb = serie[b]
            for k in range(1, hueco):
                t = k / hueco; completa[a + k] = (xa + t*(xb-xa), ya + t*(yb-ya))
    completa[fs[-1]] = serie[fs[-1]]
    fs2 = sorted(completa); arr = np.array([completa[f] for f in fs2]); suav, h = {}, VENTANA // 2
    for i, f in enumerate(fs2):
        lo, hi = max(0, i-h), min(len(arr), i+h+1); suav[f] = tuple(arr[lo:hi].mean(axis=0))
    return suav

tray_suave = {key: suavizar_y_rellenar(serie) for key, serie in tray.items()}

# Árbitros aparte (los reales tras el post-filtro)
arb_tray = defaultdict(dict)
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") == "arbitro" and en_campo(d.get("campo")):
            arb_tray[d["id"]][f_idx] = tuple(d["campo"])
arb_suave = {aid: suavizar_y_rellenar(s) for aid, s in arb_tray.items()}

# Balón: recoger detecciones, filtrar saltos imposibles e interpolar huecos cortos
balon_raw = {}
for f_idx, dets in enumerate(frames):
    for d in dets:
        if d.get("rol") == "balon" and en_campo(d.get("campo")):
            balon_raw[f_idx] = tuple(d["campo"])

MAX_HUECO_BALON = 25   # interpolar huecos de hasta 25 frames (1s); más largos se dejan vacíos
MAX_VEL_BALON = 4.0    # m/frame: descartar saltos imposibles del balón (detección errónea)

if balon_raw:
    fs = sorted(balon_raw)
    balon_filt = {fs[0]: balon_raw[fs[0]]}
    f_ant, p_ant = fs[0], balon_raw[fs[0]]
    for f in fs[1:]:
        p = balon_raw[f]
        vel = np.hypot(p[0]-p_ant[0], p[1]-p_ant[1]) / max(f-f_ant, 1)
        if vel <= MAX_VEL_BALON:
            balon_filt[f] = p
            f_ant, p_ant = f, p
else:
    balon_filt = {}

MAX_DIST_INTERP_BALON = 8.0   # m: solo interpolar si el balón reaparece a menos de 8m
                              # (movimiento continuo). Si reaparece lejos, no sabemos la
                              # trayectoria -> se oculta en el hueco en vez de inventar línea recta.
balon_tray = {}
if balon_filt:
    fs = sorted(balon_filt)
    for a, b in zip(fs, fs[1:]):
        balon_tray[a] = balon_filt[a]
        hueco = b - a
        xa, ya = balon_filt[a]; xb, yb = balon_filt[b]
        dist = np.hypot(xb - xa, yb - ya)
        # interpolar SOLO si el hueco es corto Y el balón reaparece cerca
        if 1 < hueco <= MAX_HUECO_BALON and dist <= MAX_DIST_INTERP_BALON:
            for k in range(1, hueco):
                t = k / hueco
                balon_tray[a + k] = (xa + t*(xb-xa), ya + t*(yb-ya))
        # si reaparece lejos, NO se interpola: el balón simplemente no aparece en el hueco
    balon_tray[fs[-1]] = balon_filt[fs[-1]]

# Reindexar por frame para el render
pos_por_frame = defaultdict(list)
for key, serie in tray_suave.items():
    eq, rol = meta[key]["equipo"], meta[key]["rol"]
    for f, (x, y) in serie.items():
        pos_por_frame[f].append((x, y, eq, rol))
arb_por_frame = defaultdict(list)
for aid, serie in arb_suave.items():
    for f, (x, y) in serie.items():
        arb_por_frame[f].append((x, y))

print(f"Identidades jugador/portero: {len(tray_suave)} | Árbitros: {len(arb_suave)}")
print(f"Posiciones: {sum(len(s) for s in tray.values())} brutas -> {sum(len(s) for s in tray_suave.values())} procesadas")
print(f"Balón: {len(balon_raw)} detectado -> {len(balon_filt)} tras filtro -> {len(balon_tray)} tras interpolar "
      f"({100*len(balon_tray)/N_FRAMES:.0f}% cobertura)")

Identidades jugador/portero: 24 | Árbitros: 2
Posiciones: 14969 brutas -> 15099 procesadas
Balón: 534 detectado -> 530 tras filtro -> 639 tras interpolar (85% cobertura)


In [4]:
cap = cv2.VideoCapture(str(VIDEO_IN)); fps = cap.get(cv2.CAP_PROP_FPS); cap.release()
col_eq = {eq: (np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}

OUTPUT_VIDEO = REPO_ROOT / "outputs" / f"pizarra_fase5_{NOMBRE}.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)
PIZARRA_W, PIZARRA_H = 1100, 800
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), fps, (PIZARRA_W, PIZARRA_H))

for f in range(N_FRAMES):
    fig, ax = plt.subplots(figsize=(11, 8), dpi=100)
    dibujar_campo(ax)
    for (x, y, eq, rol) in pos_por_frame.get(f, []):
        c = col_eq.get(eq, (0.5, 0.5, 0.5))
        ax.scatter(x, y, marker="o", c=[c], s=160, edgecolors="white", linewidth=1.5, zorder=5)
    for (x, y) in arb_por_frame.get(f, []):
        ax.scatter(x, y, marker="^", c="black", s=130, edgecolors="white", linewidth=1.2, zorder=5)
    if f in balon_tray:
        bx, by = balon_tray[f]
        ax.scatter(bx, by, c="yellow", s=90, edgecolors="black", linewidth=1.5, zorder=6)
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    img = cv2.resize(img, (PIZARRA_W, PIZARRA_H))
    writer.write(img); plt.close(fig)
    if f % 50 == 0: print(f"Pizarra: {f}/{N_FRAMES} frames...")

writer.release()
print(f"\nPizarra guardada en: {OUTPUT_VIDEO}")

Pizarra: 0/749 frames...
Pizarra: 50/749 frames...
Pizarra: 100/749 frames...
Pizarra: 150/749 frames...
Pizarra: 200/749 frames...
Pizarra: 250/749 frames...
Pizarra: 300/749 frames...
Pizarra: 350/749 frames...
Pizarra: 400/749 frames...
Pizarra: 450/749 frames...
Pizarra: 500/749 frames...
Pizarra: 550/749 frames...
Pizarra: 600/749 frames...
Pizarra: 650/749 frames...
Pizarra: 700/749 frames...

Pizarra guardada en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\pizarra_fase5_A1606b0e6_0_62.mp4
